# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [2]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [4]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [5]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [6]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [7]:
from pyspark.sql.functions import monotonically_increasing_id

# Ajout d'une colonne 'trip_id' comme clé unique
df_trips = df_trips.withColumn("trip_id", monotonically_increasing_id())

# Afficher les premières lignes pour vérifier
df_trips.select("trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count").show(5)

+-----------+--------------------+---------------------+---------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|
+-----------+--------------------+---------------------+---------------+
|94489280512| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|
|94489280513| 2019-01-01 00:59:47|  2019-01-01 01:18:59|            1.0|
|94489280514| 2018-12-21 13:48:30|  2018-12-21 13:52:40|            3.0|
|94489280515| 2018-11-28 15:52:25|  2018-11-28 15:55:45|            5.0|
|94489280516| 2018-11-28 15:56:57|  2018-11-28 15:58:33|            5.0|
+-----------+--------------------+---------------------+---------------+
only showing top 5 rows


In [8]:
from pyspark.sql.functions import avg, max

# 1. Le trajet avec le plus de passagers (on affiche toutes les colonnes ou quelques colonnes clés)
print("--- Trajet(s) avec le plus de passagers ---")
max_passengers = df_trips.agg(max("passenger_count")).collect()[0][0]
df_trips.filter(df_trips.passenger_count == max_passengers).select("trip_id", "passenger_count", "tpep_pickup_datetime", "trip_distance").show()

# 2. La moyenne du nombre de passagers
print("--- Nombre moyen de passagers ---")
df_trips.select(avg("passenger_count").alias("avg_passenger_count")).show()

--- Trajet(s) avec le plus de passagers ---
+-----------+---------------+--------------------+-------------+
|    trip_id|passenger_count|tpep_pickup_datetime|trip_distance|
+-----------+---------------+--------------------+-------------+
|94490230468|            9.0| 2019-01-05 13:12:29|          0.0|
|94490576799|            9.0| 2019-01-07 03:19:36|          0.0|
|94491292610|            9.0| 2019-01-10 00:43:10|          0.0|
|94492164507|            9.0| 2019-01-13 04:13:24|          0.0|
|94493815219|            9.0| 2019-01-19 16:45:25|          0.0|
|94494132737|            9.0| 2019-01-21 03:46:51|          0.0|
|94494278302|            9.0| 2019-01-21 19:20:28|          0.0|
|94496567195|            9.0| 2019-01-30 18:34:12|          0.0|
|94496654388|            9.0| 2019-01-30 22:17:51|        13.38|
+-----------+---------------+--------------------+-------------+

--- Nombre moyen de passagers ---
+-------------------+
|avg_passenger_count|
+-------------------+
| 1.567031

In [9]:
from pyspark.sql.functions import col, unix_timestamp, max as spark_max, min as spark_min

# 1. Plus court / plus long par distance
print("--- Plus court / plus long par distance ---")
df_trips.select("trip_id", "trip_distance", "tpep_pickup_datetime", "tpep_dropoff_datetime") \
    .orderBy(col("trip_distance").asc()).show(1) # Le plus court

df_trips.select("trip_id", "trip_distance", "tpep_pickup_datetime", "tpep_dropoff_datetime") \
    .orderBy(col("trip_distance").desc()).show(1) # Le plus long

# 2. Calcul de la durée du trajet en secondes, puis plus court / plus long par temps
df_trips_with_duration = df_trips.withColumn(
    "trip_duration_seconds", 
    unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")
)

print("--- Plus court par temps ---")
df_trips_with_duration.select("trip_id", "trip_duration_seconds", "trip_distance", "tpep_pickup_datetime", "tpep_dropoff_datetime") \
    .orderBy(col("trip_duration_seconds").asc()).show(1)

print("--- Plus long par temps ---")
df_trips_with_duration.select("trip_id", "trip_duration_seconds", "trip_distance", "tpep_pickup_datetime", "tpep_dropoff_datetime") \
    .orderBy(col("trip_duration_seconds").desc()).show(1)

--- Plus court / plus long par distance ---
+-----------+-------------+--------------------+---------------------+
|    trip_id|trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|
+-----------+-------------+--------------------+---------------------+
|94489280514|          0.0| 2018-12-21 13:48:30|  2018-12-21 13:52:40|
+-----------+-------------+--------------------+---------------------+
only showing top 1 row
+-----------+-------------+--------------------+---------------------+
|    trip_id|trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|
+-----------+-------------+--------------------+---------------------+
|94495354603|        831.8| 2019-01-25 21:56:39|  2019-01-25 22:06:08|
+-----------+-------------+--------------------+---------------------+
only showing top 1 row
--- Plus court par temps ---
+-----------+---------------------+-------------+--------------------+---------------------+
|    trip_id|trip_duration_seconds|trip_distance|tpep_pickup_datetime|tpep_dro

In [11]:
from pyspark.sql.functions import to_date, hour, date_format, count, desc, asc 
# Extraire la date et l'heure du pickup
df_trips_time = df_trips.withColumn("pickup_date", to_date("tpep_pickup_datetime")) \
                        .withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
                        .withColumn("day_of_week", date_format("tpep_pickup_datetime", "E")) # Ex: Mon, Tue...

# 1. Busiest / Slowest single day
print("--- Jour le plus chargé ---")
df_trips_time.groupBy("pickup_date").agg(count("*").alias("trip_count")) \
             .orderBy(desc("trip_count")).show(1)

print("--- Jour le plus calme ---")
df_trips_time.groupBy("pickup_date").agg(count("*").alias("trip_count")) \
             .orderBy(asc("trip_count")).show(1)

# 2. Busiest / Slowest hour of the day
print("--- Heure de la journée la plus chargée ---")
df_trips_time.groupBy("pickup_hour").agg(count("*").alias("trip_count")) \
             .orderBy(desc("trip_count")).show(1)

print("--- Heure de la journée la plus calme ---")
df_trips_time.groupBy("pickup_hour").agg(count("*").alias("trip_count")) \
             .orderBy(asc("trip_count")).show(1)

# 3. Busiest / Slowest day of the week
print("--- Jour de la semaine le plus chargé / calme ---")
df_trips_time.groupBy("day_of_week").agg(count("*").alias("trip_count")) \
             .orderBy(desc("trip_count")).show(7)

--- Jour le plus chargé ---
+-----------+----------+
|pickup_date|trip_count|
+-----------+----------+
| 2019-01-25|    292499|
+-----------+----------+
only showing top 1 row
--- Jour le plus calme ---
+-----------+----------+
|pickup_date|trip_count|
+-----------+----------+
| 2019-05-20|         1|
+-----------+----------+
only showing top 1 row
--- Heure de la journée la plus chargée ---
+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|         18|    515390|
+-----------+----------+
only showing top 1 row
--- Heure de la journée la plus calme ---
+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|          4|     61424|
+-----------+----------+
only showing top 1 row
--- Jour de la semaine le plus chargé / calme ---
+-----------+----------+
|day_of_week|trip_count|
+-----------+----------+
|        Thu|   1357043|
|        Wed|   1265264|
|        Tue|   1209084|
|        Fri|   1087215|
|        Sat|   1009985|
|        Mon|    

In [12]:
from pyspark.sql.functions import max as spark_max

# 1. Corrélation entre la distance / passagers et le pourboire
corr_distance = df_trips.stat.corr("trip_distance", "tip_amount")
corr_passengers = df_trips.stat.corr("passenger_count", "tip_amount")

print(f"Corrélation (Distance vs Tip) : {corr_distance}")
print(f"Corrélation (Passenger Count vs Tip) : {corr_passengers}")

# 2. Le plus haut frais "extra"
print("--- Trajet avec le frais 'extra' le plus élevé ---")
max_extra = df_trips.agg(spark_max("extra")).collect()[0][0]
df_trips.filter(df_trips.extra == max_extra).select("trip_id", "extra", "trip_distance", "tpep_pickup_datetime").show()

Corrélation (Distance vs Tip) : 0.5269200663652668
Corrélation (Passenger Count vs Tip) : 0.004431051585116288
--- Trajet avec le frais 'extra' le plus élevé ---
+-----------+------+-------------+--------------------+
|    trip_id| extra|trip_distance|tpep_pickup_datetime|
+-----------+------+-------------+--------------------+
|94494603995|535.38|          0.0| 2019-01-23 08:58:09|
+-----------+------+-------------+--------------------+



### Analyse des valeurs aberrantes (Outliers) observées :
1. **Incohérences temporelles / Dates erronées** : Le dataset cible normalement janvier 2019, mais nous avons trouvé des trajets datant de fin 2018 (ex: déc 2018) ou de mai 2019.
2. **Durées de trajet aberrantes** : 
   - Présence de durées de trajet négatives (ex: -5 056 830 secondes), causées par une date de *dropoff* antérieure à la date de *pickup*.
   - Des durées de trajet extrêmement longues (plusieurs semaines ou mois) pour des distances très courtes (ex: 1,2 miles en 1 mois).
3. **Distances et Passagers suspects** : 
   - Des trajets avec une distance de 0.0 miles mais affichant des frais "extra" faramineux (ex: 535.38 $ de frais "extra" pour 0.0 mile) ou 9 passagers.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [15]:
import requests

# 1. Télécharger et charger la table de correspondance des zones
zone_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
zone_response = requests.get(zone_url)

zone_file = "taxi_zone_lookup.csv"
if zone_response.status_code == 200:
    with open(zone_file, "wb") as f:
        f.write(zone_response.content)

df_zones = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(zone_file)

# 2. Joindre pour récupérer le Borough de départ (PULocationID)
df_trips_joined = df_trips.join(
    df_zones, 
    df_trips.PULocationID == df_zones.LocationID, 
    "left"
).withColumnRenamed("Borough", "pickup_borough") \
 .withColumnRenamed("Zone", "pickup_zone") \
 .drop("LocationID", "service_zone")

# Afficher un aperçu
df_trips_joined.select("trip_id", "pickup_borough", "pickup_zone", "trip_distance").show(5)

+-----------+--------------+--------------------+-------------+
|    trip_id|pickup_borough|         pickup_zone|trip_distance|
+-----------+--------------+--------------------+-------------+
|94489280512|     Manhattan|    Manhattan Valley|          1.5|
|94489280513|     Manhattan|Upper West Side S...|          2.6|
|94489280514|     Manhattan|Upper East Side N...|          0.0|
|94489280515|        Queens|Queensbridge/Rave...|          0.0|
|94489280516|        Queens|Queensbridge/Rave...|          0.0|
+-----------+--------------+--------------------+-------------+
only showing top 5 rows


In [16]:
from pyspark.sql.functions import count, avg, desc

# 1. Joindre aussi pour le dropoff borough pour pouvoir répondre aux deux
df_trips_fully_joined = df_trips_joined.join(
    df_zones, 
    df_trips.DOLocationID == df_zones.LocationID, 
    "left"
).withColumnRenamed("Borough", "dropoff_borough") \
 .withColumnRenamed("Zone", "dropoff_zone") \
 .drop("LocationID", "service_zone")

# 2. Borough avec le plus de Pickups
print("--- Pickups par Borough ---")
df_trips_fully_joined.groupBy("pickup_borough").agg(count("*").alias("pickup_count")) \
                     .orderBy(desc("pickup_count")).show()

# 3. Borough avec le plus de Dropoffs
print("--- Dropoffs par Borough ---")
df_trips_fully_joined.groupBy("dropoff_borough").agg(count("*").alias("dropoff_count")) \
                     .orderBy(desc("dropoff_count")).show()

# 4. Distance moyenne et tarif moyen par Pickup Borough
print("--- Distance et Tarif moyens par Pickup Borough ---")
df_trips_fully_joined.groupBy("pickup_borough") \
                     .agg(avg("trip_distance").alias("avg_distance"), 
                          avg("total_amount").alias("avg_fare")) \
                     .orderBy(desc("avg_distance")).show()

--- Pickups par Borough ---
+--------------+------------+
|pickup_borough|pickup_count|
+--------------+------------+
|     Manhattan|     6950965|
|        Queens|      471173|
|       Unknown|      159815|
|      Brooklyn|       91905|
|         Bronx|       18062|
|           N/A|        3890|
|           EWR|         446|
| Staten Island|         361|
+--------------+------------+

--- Dropoffs par Borough ---
+---------------+-------------+
|dropoff_borough|dropoff_count|
+---------------+-------------+
|      Manhattan|      6817355|
|         Queens|       340972|
|       Brooklyn|       301105|
|        Unknown|       149097|
|          Bronx|        58085|
|            N/A|        16904|
|            EWR|        10914|
|  Staten Island|         2185|
+---------------+-------------+

--- Distance et Tarif moyens par Pickup Borough ---
+--------------+------------------+------------------+
|pickup_borough|      avg_distance|          avg_fare|
+--------------+------------------+

In [17]:
from pyspark.sql.functions import hour, date_format, desc, row_number, count, min as spark_min, max as spark_max
from pyspark.sql.window import Window

# Préparer les colonnes heure et jour de la semaine si ce n'est pas déjà fait
df_p2 = df_trips_fully_joined.withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
                               .withColumn("day_of_week", date_format("tpep_pickup_datetime", "E"))

# 1. Heure la plus chargée par Pickup Borough
print("--- Heure la plus chargée par Pickup Borough ---")
hour_window = Window.partitionBy("pickup_borough").orderBy(desc("trip_count"))
df_p2.groupBy("pickup_borough", "pickup_hour").agg(count("*").alias("trip_count")) \
     .withColumn("rn", row_number().over(hour_window)) \
     .filter("rn == 1").show()

# 2. Tarif le plus élevé et son borough associé
print("--- Tarif le plus élevé ---")
max_amount = df_p2.agg(spark_max("total_amount")).collect()[0][0]
df_p2.filter(df_p2.total_amount == max_amount) \
     .select("trip_id", "total_amount", "pickup_borough", "dropoff_borough").show()

# 3. Tarif le plus bas et ses boroughs associés
print("--- Tarif le plus bas ---")
min_amount = df_p2.agg(spark_min("total_amount")).collect()[0][0]
df_p2.filter(df_p2.total_amount == min_amount) \
     .select("trip_id", "total_amount", "pickup_borough", "dropoff_borough").show(5)

--- Heure la plus chargée par Pickup Borough ---
+--------------+-----------+----------+---+
|pickup_borough|pickup_hour|trip_count| rn|
+--------------+-----------+----------+---+
|         Bronx|          7|      1803|  1|
|      Brooklyn|          8|      6935|  1|
|           EWR|         15|        54|  1|
|     Manhattan|         18|    471539|  1|
|           N/A|         19|       214|  1|
|        Queens|         16|     29885|  1|
| Staten Island|          8|        36|  1|
|       Unknown|         18|     10751|  1|
+--------------+-----------+----------+---+

--- Tarif le plus élevé ---
+-----------+------------+--------------+---------------+
|    trip_id|total_amount|pickup_borough|dropoff_borough|
+-----------+------------+--------------+---------------+
|94491780167|   623261.66|     Manhattan|      Manhattan|
+-----------+------------+--------------+---------------+

--- Tarif le plus bas ---
+-----------+------------+--------------+---------------+
|    trip_id|total_

In [18]:
# 1. Télécharger les données de janvier 2025
url_2025 = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet'
resp_2025 = requests.get(url_2025)

file_2025 = "yellow_tripdata_2025-01.parquet"
if resp_2025.status_code == 200:
    with open(file_2025, "wb") as f:
        f.write(resp_2025.content)

# 2. Charger le DataFrame 2025
df_2025 = spark.read.parquet(file_2025)

# 3. Comparaison des moyennes (2019 vs 2025)
print("--- Métriques Janvier 2019 ---")
df_trips.select(avg("trip_distance").alias("avg_dist_2019"), avg("total_amount").alias("avg_fare_2019")).show()

print("--- Métriques Janvier 2025 ---")
df_2025.select(avg("trip_distance").alias("avg_dist_2025"), avg("total_amount").alias("avg_fare_2025")).show()

--- Métriques Janvier 2019 ---
+------------------+-----------------+
|     avg_dist_2019|    avg_fare_2019|
+------------------+-----------------+
|2.8301461681153532|15.81065134371489|
+------------------+-----------------+

--- Métriques Janvier 2025 ---
+-----------------+------------------+
|    avg_dist_2025|     avg_fare_2025|
+-----------------+------------------+
|5.855126178843539|25.611291697280986|
+-----------------+------------------+



### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [20]:
# 1. Enregistrer les DataFrames comme vues temporaires SQL
df_trips.createOrReplaceTempView("nyc_trips")
df_zones.createOrReplaceTempView("taxi_zones")

# 2. Question 1 : Nombre moyen de passagers (Simple agrégation SQL)
print("--- Q1 (SQL) : Nombre moyen de passagers ---")
spark.sql("""
    SELECT avg(passenger_count) as avg_passengers 
    FROM nyc_trips
""").show()

# 3. Question 2 : Heure de la journée la plus chargée (Group by + Order by)
print("--- Q2 (SQL) : Heure de la journée la plus chargée ---")
spark.sql("""
    SELECT hour(tpep_pickup_datetime) as pickup_hour, count(*) as trip_count 
    FROM nyc_trips 
    GROUP BY pickup_hour 
    ORDER BY trip_count DESC 
    LIMIT 1
""").show()

# 4. Question 3 : Distance moyenne et tarif moyen par Borough (Jointure SQL obligatoire)
print("--- Q3 (SQL) : Distance et tarif moyens par Borough (avec jointure) ---")
spark.sql("""
    SELECT z.Borough as pickup_borough, 
           avg(t.trip_distance) as avg_distance, 
           avg(t.total_amount) as avg_fare
    FROM nyc_trips t
    JOIN taxi_zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY avg_distance DESC
""").show()


--- Q1 (SQL) : Nombre moyen de passagers ---
+------------------+
|    avg_passengers|
+------------------+
|1.5670317144945614|
+------------------+

--- Q2 (SQL) : Heure de la journée la plus chargée ---
+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|         18|    515390|
+-----------+----------+

--- Q3 (SQL) : Distance et tarif moyens par Borough (avec jointure) ---
+--------------+------------------+------------------+
|pickup_borough|      avg_distance|          avg_fare|
+--------------+------------------+------------------+
| Staten Island|12.503601108033246| 53.58659279778373|
|        Queens|11.283218499361993| 44.45193604055996|
|         Bronx| 7.233194552098303|29.306521979843616|
|      Brooklyn| 4.787677275447492| 21.61839747567109|
|           N/A| 3.193850899742941| 69.54899485860886|
|           EWR| 2.641098654708519| 92.78352017937229|
|       Unknown| 2.415464130400774| 18.16889928978967|
|     Manhattan|2.2286693358402596|13.66614106

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing